# 99_run — Black-Litterman 실험 실행

**역할**: bl_config.py에 정의된 모든 실험을 walk-forward로 실행하고 결과를 `results/` 폴더에 저장.

## 실행 순서
1. 데이터 로드
2. Dispatcher 함수 정의 (슬롯별 config → 함수 분기)
3. walk_forward() 함수 정의
4. 전체 실험 실행 → pkl 저장

> 분석/시각화는 **99_analyze.ipynb**에서.

In [28]:
import pandas as pd
import numpy as np
import pickle
import warnings
import platform
from pathlib import Path

import matplotlib
matplotlib.use('Agg')

warnings.filterwarnings('ignore')

from bl_functions import (
    compute_sigma, compute_daily_slice, compute_pi,
    build_P,
    compute_Q_fixed, compute_Q_lambda, compute_Q_inv_lambda, compute_Q_raw_lam, compute_Q_vol_spread,
    compute_Q_ff3_paper,
    compute_Q_ff3, compute_Q_spread, compute_Q_regime,
    compute_omega_he, compute_omega_scaled, compute_omega_rmse,
    black_litterman, optimize_portfolio,
    compute_turnover, apply_tc, compute_metrics,
)
from bl_config import EXPERIMENTS, get_changed_slots

BASE_DIR    = Path.cwd()
DATA_DIR    = BASE_DIR / 'data'
RESULTS_DIR = BASE_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

assert DATA_DIR.exists(), f'data/ 폴더 없음. 01_DataCollection.ipynb 먼저 실행하세요.\n경로: {DATA_DIR}'

# ── 공통 파라미터 ──────────────────────────────────────────────
TRAIN_WINDOW  = 60
THRESH_DAILY  = 0.9
TAU           = 0.1
PCT_GROUP     = 0.30
START_PRED    = '2010-01-01'

print(f'설정 완료')
print(f'  DATA_DIR    : {DATA_DIR}')
print(f'  RESULTS_DIR : {RESULTS_DIR}')
print(f'  실험 수     : {len(EXPERIMENTS)}개')

설정 완료
  DATA_DIR    : /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/final/data
  RESULTS_DIR : /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/final/results
  실험 수     : 175개


In [29]:
# ── 데이터 로드 ───────────────────────────────────────────────
panel     = pd.read_csv(DATA_DIR / 'monthly_panel.csv', parse_dates=['date'])
panel     = panel.set_index(['date', 'ticker'])
daily_ret = pd.read_pickle(DATA_DIR / 'daily_returns.pkl')

all_dates  = panel.index.get_level_values('date').unique().sort_values()
pred_dates = all_dates[all_dates >= START_PRED]

spy_series = panel['spy_ret'].groupby(level='date').first()
rf_series  = panel['rf_1m'].groupby(level='date').first()
ret_pivot  = panel['ret_1m'].unstack('ticker')   # FF3 Q 계산용

# FF3 팩터
import io, re, zipfile, requests
FF3_PATH = DATA_DIR / 'ff3_monthly.csv'
if FF3_PATH.exists():
    ff3 = pd.read_csv(FF3_PATH, index_col='date', parse_dates=True)
else:
    print('FF3 다운로드 중...')
    url  = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_CSV.zip'
    resp = requests.get(url, timeout=60)
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        raw = zf.read(zf.namelist()[0]).decode('utf-8', errors='ignore')
    lines = raw.splitlines()
    start = next(i for i, l in enumerate(lines) if re.match(r'^\s*\d{6}\s*,', l))
    end   = next((i for i in range(start, len(lines)) if not re.match(r'^\s*\d{6}\s*,', lines[i])), len(lines))
    df    = pd.read_csv(io.StringIO('\n'.join(lines[start-1:end])))
    df.columns = [c.strip() for c in df.columns]
    dc = df.columns[0]
    df[dc] = pd.to_datetime(df[dc].astype(str), format='%Y%m') + pd.offsets.MonthEnd(0)
    ff3 = df.rename(columns={dc:'date','Mkt-RF':'mkt_rf','SMB':'smb','HML':'hml','RF':'rf'}).set_index('date').astype(float) / 100.0
    ff3.to_csv(FF3_PATH)

print(f'패널       : {panel.shape}')
print(f'일별 수익률: {daily_ret.shape}')
print(f'예측 기간  : {pred_dates[0].date()} ~ {pred_dates[-1].date()} ({len(pred_dates)}개월)')

패널       : (108135, 11)
일별 수익률: (5594, 835)
예측 기간  : 2010-01-31 ~ 2025-12-31 (192개월)


In [30]:
# ── LSTM 예측값 로드 (없으면 None) ────────────────────────────
# bl_config.py의 BASELINE['lstm_pred_path']에 경로 지정
# 파일이 없으면 LSTM 관련 실험은 자동 스킵

from bl_config import BASELINE

_lstm_path = Path(BASELINE.get('lstm_pred_path', ''))
if _lstm_path.exists():
    _raw = pd.read_csv(_lstm_path, parse_dates=['date'])
    _raw['vol_pred'] = np.exp(_raw['y_pred_ensemble'])          # log-RV → 실제 변동성
    _raw['abs_err']  = (_raw['y_pred_ensemble'] - _raw['y_true']).abs()

    # 원본 (date, ticker) 인덱스 — omega_rmse 계산용
    lstm_preds = _raw.set_index(['date', 'ticker'])

    # ── 월별 피벗 (Phase3 get_monthly_pred 방식) ──────────────
    # 각 종목·월 기준 마지막 예측값 → 시장 거래일 월말 매핑
    _m = _raw.copy()
    _m['month'] = _m['date'].dt.to_period('M')
    _last = _m.groupby(['ticker', 'month'])['vol_pred'].last().reset_index()
    _month_map = {pd.Timestamp(d).to_period('M'): pd.Timestamp(d) for d in pred_dates}
    _last['pred_date'] = _last['month'].map(_month_map)
    _last = _last.dropna(subset=['pred_date'])
    lstm_pred_monthly = _last.pivot_table(
        index='pred_date', columns='ticker', values='vol_pred')

    print(f'LSTM 예측 로드: {len(_raw):,}행')
    print(f'LSTM 월별 피벗: {lstm_pred_monthly.shape[0]}개월 × {lstm_pred_monthly.shape[1]}종목')
    LSTM_AVAILABLE = True
else:
    lstm_preds        = None
    lstm_pred_monthly = None
    LSTM_AVAILABLE    = False
    print(f'⚠  LSTM 예측 파일 없음 → lstm 관련 실험 스킵')
    print(f'   경로: {_lstm_path}')

LSTM 예측 로드: 2,468,883행
LSTM 월별 피벗: 192개월 × 613종목


In [ ]:
# ══════════════════════════════════════════════════════════════
# Dispatcher 함수 (config → 실제 함수 호출)
# 새 방식 추가 시 해당 elif만 추가
# ══════════════════════════════════════════════════════════════

def get_vol_series(cfg, month_df, pred_date):
    """P 행렬에 쓸 변동성 시리즈 반환."""
    mode = cfg.get('p_mode', 'trailing_vol21')

    if mode == 'trailing_vol21':
        return month_df['vol_21d']

    elif mode == 'trailing_vol252':
        return month_df['vol_252d']

    elif mode == 'lstm_predicted':
        if lstm_pred_monthly is None or pred_date not in lstm_pred_monthly.index:
            return month_df['vol_21d']
        pred_slice = lstm_pred_monthly.loc[pred_date].dropna()
        vol = month_df['vol_21d'].copy()
        common = vol.index.intersection(pred_slice.index)
        if len(common) >= 5:
            vol[common] = pred_slice[common]
        return vol

    else:
        raise ValueError(f'p_mode={mode!r} 미지원')


def get_prior_weights(cfg, valid_tix, mcap, vol=None):
    """prior 시장가중치 w_mkt 반환. capm_rp는 vol(trailing) 필요."""
    prior = cfg.get('prior', 'capm_mcap')
    if prior == 'capm_mcap':
        return (mcap / mcap.sum()).reindex(valid_tix).fillna(0)
    elif prior == 'capm_eq':
        n = len(valid_tix)
        return pd.Series(1.0 / n, index=valid_tix)
    elif prior == 'capm_rp':
        # Risk Parity: 1/σ 정규화 (자산 독립 가정, 단순 RP)
        if vol is None:
            raise ValueError('capm_rp requires vol parameter')
        inv_vol = 1.0 / vol.replace(0, np.nan).dropna()
        w = inv_vol / inv_vol.sum()
        return w.reindex(valid_tix).fillna(0)
    else:
        raise ValueError(f'prior={prior!r} 미지원')


def get_Q(cfg, P, valid_tix, train_dates, pred_date, all_d, lam=2.5, spy_excess=0.0, sigma2_mkt=0.001):
    """Q 값 반환. q_mode='capm'/'none'은 walk_forward에서 직접 처리."""
    mode = cfg.get('q_mode', 'fixed')

    if mode == 'fixed':
        return compute_Q_fixed(cfg.get('q_value', 0.003))

    elif mode == 'ff3_regression':
        thresh_m    = int(len(train_dates) * 0.7)
        monthly_sl  = ret_pivot.reindex(index=train_dates, columns=valid_tix).dropna(axis=1, thresh=thresh_m)
        ff3_train   = ff3.reindex(train_dates)
        rf_train    = rf_series.reindex(train_dates)
        return compute_Q_ff3(P.reindex(monthly_sl.columns).fillna(0), monthly_sl, ff3_train, rf_train)

    elif mode == 'ff3_paper':
        # Q만 반환, Ω는 walk-forward 루프에서 전월 오차²로 계산
        thresh_m    = int(len(train_dates) * 0.7)
        monthly_sl  = ret_pivot.reindex(index=train_dates, columns=valid_tix).dropna(axis=1, thresh=thresh_m)
        ff3_train   = ff3.reindex(train_dates)
        rf_train    = rf_series.reindex(train_dates)
        return compute_Q_ff3_paper(P.reindex(monthly_sl.columns).fillna(0), monthly_sl, ff3_train, rf_train)

    elif mode == 'realized_spread':
        panel_train = panel[
            (panel.index.get_level_values('date') >= train_dates[0]) &
            (panel.index.get_level_values('date') <= train_dates[-1])
        ]
        return compute_Q_spread(panel_train, pct=PCT_GROUP)

    elif mode == 'regime':
        spy_train = spy_series.reindex(train_dates)
        return compute_Q_regime(spy_train, cfg.get('q_regime_table', {}))

    elif mode == 'lambda':
        return compute_Q_lambda(lam, cfg.get('q_value', 0.003), cfg.get('lam_mean', 2.5))

    elif mode == 'inv_lambda':
        return compute_Q_inv_lambda(lam, cfg.get('q_value', 0.003), cfg.get('lam_mean', 2.5))

    elif mode == 'raw_lam':
        return compute_Q_raw_lam(spy_excess, sigma2_mkt, cfg.get('q_value', 0.003), cfg.get('lam_mean', 2.5))

    elif mode == 'vol_spread':
        # LSTM 예측 vol 격차 기반 동적 Q (사용자 추가)
        if lstm_pred_monthly is None or pred_date not in lstm_pred_monthly.index:
            return cfg.get('q_value', 0.003)  # LSTM 없으면 fallback to fixed

        # 현재 시점 예측 vol
        vol_pred_curr = lstm_pred_monthly.loc[pred_date].dropna()

        # 과거 spread들 (train_dates 구간) — look-ahead bias 없음
        past_spreads = []
        for past_dt in train_dates:
            if past_dt not in lstm_pred_monthly.index:
                continue
            vp = lstm_pred_monthly.loc[past_dt].dropna()
            if len(vp) < 20:
                continue
            n_grp = max(1, int(len(vp) * PCT_GROUP))
            sorted_vp = vp.sort_values()
            low_m  = sorted_vp.iloc[:n_grp].mean()
            high_m = sorted_vp.iloc[-n_grp:].mean()
            past_spreads.append(high_m - low_m)
        if not past_spreads:
            return cfg.get('q_value', 0.003)
        spread_ref = float(np.median(past_spreads))

        return compute_Q_vol_spread(
            P, vol_pred_curr,
            cfg.get('q_value', 0.003),
            spread_ref,
        )

    else:
        raise ValueError(f'q_mode={mode!r} 미지원')


# abs_err 실제 중앙값 (~0.24) — inf 값 존재하므로 median 사용
_OMEGA_BASE_RMSE = 0.24

def get_omega(cfg, P, Sigma, pred_date):
    """Omega 값 반환."""
    mode = cfg.get('omega_mode', 'he_litterman')

    if mode == 'he_litterman':
        return compute_omega_he(P, Sigma, TAU)

    elif mode == 'scaled':
        return compute_omega_scaled(P, Sigma, TAU, cfg.get('omega_scale', 1.0))

    elif mode == 'rmse':
        if lstm_preds is None:
            return compute_omega_he(P, Sigma, TAU)
        cutoff = pred_date - pd.DateOffset(months=12)
        recent = lstm_preds[
            (lstm_preds.index.get_level_values('date') >= cutoff) &
            (lstm_preds.index.get_level_values('date') <= pred_date)
        ]
        if len(recent) > 0:
            # inf/nan 필터링 후 median 사용 (mean은 inf 오염 가능)
            err_vals = recent['abs_err'].replace([np.inf, -np.inf], np.nan).dropna()
            pred_rmse = float(err_vals.median()) if len(err_vals) > 0 else _OMEGA_BASE_RMSE
        else:
            pred_rmse = _OMEGA_BASE_RMSE
        return compute_omega_rmse(P, Sigma, TAU, pred_rmse, base_rmse=_OMEGA_BASE_RMSE)

    # 'ff3_paper' 모드는 walk_forward에서 직접 처리 (직전월 예측오차²) — get_omega로 안 옴

    else:
        raise ValueError(f'omega_mode={mode!r} 미지원')


print('Dispatcher 함수 정의 완료')

In [32]:
import time

# ══════════════════════════════════════════════════════════════
# 월별 공유 데이터 캐시 (모든 실험에서 동일한 부분 — 한 번만 계산)
#   Sigma (LedoitWolf), month_df, mcap, train_dates, spy_excess, sigma2_mkt
#   실험별로 달라지는 것: vol_series(p_mode), P(p_weight), Q, omega, w
# ══════════════════════════════════════════════════════════════

monthly_cache = {}
_cache_t0 = time.time()

for i, pred_date in enumerate(pred_dates):
    if i % 24 == 0:
        elapsed = time.time() - _cache_t0
        pct     = (i + 1) / len(pred_dates)
        eta     = elapsed / pct * (1 - pct) if pct > 0.01 else 0
        print(f'  캐시 {pred_date.strftime("%Y-%m")} ({i+1}/{len(pred_dates)}, {pct:.0%}) | '
              f'경과 {elapsed/60:.1f}분 | ETA {eta/60:.1f}분', flush=True)
    try:
        month_df = panel.xs(pred_date, level='date').dropna(
            subset=['vol_21d', 'log_mcap', 'ret_1m'])
        if len(month_df) < 30:
            continue

        daily_slice, valid_tix = compute_daily_slice(
            pred_date, month_df.index.tolist(),
            daily_ret, TRAIN_WINDOW, THRESH_DAILY)
        if len(valid_tix) < 20:
            continue

        Sigma    = compute_sigma(daily_slice, scale=21)
        month_df = month_df.reindex(valid_tix)
        mcap     = np.exp(month_df['log_mcap'])

        idx         = all_dates.get_loc(pred_date)
        train_dates = all_dates[max(0, idx - TRAIN_WINDOW): idx]
        spy_s       = spy_series.reindex(train_dates)
        rf_s        = rf_series.reindex(train_dates)

        next_date = all_dates[idx + 1] if idx + 1 < len(all_dates) else None

        monthly_cache[pred_date] = {
            'month_df'   : month_df,
            'valid_tix'  : valid_tix,
            'Sigma'      : Sigma,
            'mcap'       : mcap,
            'train_dates': train_dates,
            'spy_excess' : float((spy_s - rf_s).mean()),
            'sigma2_mkt' : float(spy_s.var()),
            'next_date'  : next_date,
        }
    except Exception as e:
        print(f'  [캐시 에러] {pred_date.date()}: {e}')

_cache_min = (time.time() - _cache_t0) / 60
print(f'\n캐시 완료: {len(monthly_cache)}개월 / {len(pred_dates)}개월 | {_cache_min:.1f}분 소요')
print('이후 실험은 Sigma 재계산 없이 캐시에서 로드')

  캐시 2010-01 (1/192, 1%) | 경과 0.0분 | ETA 0.0분
  캐시 2012-01 (25/192, 13%) | 경과 0.0분 | ETA 0.1분
  캐시 2014-01 (49/192, 26%) | 경과 0.0분 | ETA 0.1분
  캐시 2016-01 (73/192, 38%) | 경과 0.0분 | ETA 0.1분
  캐시 2018-01 (97/192, 51%) | 경과 0.1분 | ETA 0.1분
  캐시 2020-01 (121/192, 63%) | 경과 0.1분 | ETA 0.0분
  캐시 2022-01 (145/192, 76%) | 경과 0.1분 | ETA 0.0분
  캐시 2024-01 (169/192, 88%) | 경과 0.1분 | ETA 0.0분

캐시 완료: 192개월 / 192개월 | 0.1분 소요
이후 실험은 Sigma 재계산 없이 캐시에서 로드


In [33]:
import time

# ══════════════════════════════════════════════════════════════
# walk_forward(cfg) — 실험 하나 실행 (캐시에서 Sigma 로드)
# ══════════════════════════════════════════════════════════════

def walk_forward(cfg: dict) -> dict:
    name       = cfg['name']
    tc         = cfg.get('tc', 0.001)
    max_w      = cfg.get('max_weight', 0.10)
    q_mode     = cfg.get('q_mode', 'fixed')
    p_weight   = cfg.get('p_weight', 'mcap')
    is_naive   = (q_mode == 'none')
    is_capm    = (q_mode == 'capm')

    ret_list, comp_list, meta_list = [], [], []
    spy_list, err_list = [], []
    weights_history = {}
    prev_w = None
    # omega_paper 상태 (전월 Q 예측 이력)
    _op_q_prev = None   # 직전 월 Q 예측값
    _op_P_prev = None   # 직전 월 P 벡터
    _op_errors = []     # 누적 예측 오차
    _t0 = time.time()

    for i, pred_date in enumerate(pred_dates):

        if i % 12 == 0 or i == len(pred_dates) - 1:
            elapsed = time.time() - _t0
            pct     = (i + 1) / len(pred_dates)
            eta     = elapsed / pct * (1 - pct) if pct > 0.01 else 0
            print(f'  [{name}] {pred_date.strftime("%Y-%m")} '
                  f'({i+1}/{len(pred_dates)}, {pct:.0%}) | '
                  f'경과 {elapsed/60:.1f}분 | ETA {eta/60:.1f}분', flush=True)

        if pred_date not in monthly_cache:
            continue
        c           = monthly_cache[pred_date]
        month_df    = c['month_df']
        valid_tix   = c['valid_tix']
        Sigma       = c['Sigma']
        mcap        = c['mcap']
        train_dates = c['train_dates']
        spy_excess  = c['spy_excess']
        sigma2_mkt  = c['sigma2_mkt']
        next_date   = c['next_date']

        try:
            w_mkt   = get_prior_weights(cfg, valid_tix, mcap, vol=month_df['vol_21d'])
            pi, lam = compute_pi(Sigma, w_mkt, spy_excess, sigma2_mkt)

            vol_series = get_vol_series(cfg, month_df, pred_date)
            P = build_P(
                vol_series.reindex(valid_tix).fillna(vol_series.median()),
                mcap, pct=PCT_GROUP, weighting=p_weight)

            if is_capm:
                w = optimize_portfolio(pi, Sigma.values, lam, max_w)
                mu_meta = None

            elif is_naive:
                n_g     = max(1, int(len(valid_tix) * PCT_GROUP))
                vol_v   = vol_series.reindex(valid_tix)
                low_tix = vol_v.sort_values().index[:n_g]

                if p_weight == 'mcap':
                    w_low = mcap.reindex(low_tix)
                    w_low = w_low / w_low.sum()
                elif p_weight == 'rp':
                    inv_v = (1.0 / vol_v[low_tix]).replace(0, np.nan).dropna()
                    w_low = inv_v / inv_v.sum()
                else:
                    w_low = pd.Series(1.0 / len(low_tix), index=low_tix)

                w = w_low.reindex(valid_tix).fillna(0)
                w = w.clip(upper=max_w)
                w = w / w.sum()
                mu_meta = None

            else:
                lam_q = float(np.clip(spy_excess / sigma2_mkt, 0.5, 10.0)) if sigma2_mkt > 1e-10 else 2.5
                Q     = get_Q(cfg, P, valid_tix, train_dates, pred_date, all_dates,
                              lam=lam_q, spy_excess=spy_excess, sigma2_mkt=sigma2_mkt)
                # ff3_paper: Q, Omega 동시 반환
                if cfg.get("omega_mode") == "ff3_paper":
                    # 논문 방식: 직전 월 예측오차²을 Omega로 사용
                    if _op_q_prev is not None and _op_P_prev is not None:
                        actual_p = float(
                            month_df["ret_1m"].reindex(_op_P_prev.index).fillna(0)
                            @ _op_P_prev
                        )
                        omega = max((_op_q_prev - actual_p) ** 2, 1e-8)
                    else:
                        omega = compute_omega_he(P, Sigma, TAU)  # 첫달 폴백
                    _op_q_prev = Q
                    _op_P_prev = P.copy()
                else:
                    omega = get_omega(cfg, P, Sigma, pred_date)
                mu_BL, sig_BL = black_litterman(pi, Sigma, P, Q, omega, TAU)
                w      = optimize_portfolio(mu_BL, Sigma.values, lam, max_w)
                mu_meta = Q

            actual_ret = month_df['fwd_ret_1m'].reindex(valid_tix).fillna(0)
            gross_ret  = float(w @ actual_ret)
            turnover   = compute_turnover(w, prev_w) if prev_w is not None else 0.0
            net_ret    = apply_tc(gross_ret, turnover, tc)
            r_spy      = float(spy_series.get(next_date, np.nan)) if next_date else np.nan

            ret_list.append({'date': pred_date, 'ret': net_ret, 'gross_ret': gross_ret})
            spy_list.append({'date': pred_date, 'ret': r_spy})

            vol_col      = month_df['vol_21d'].reindex(valid_tix)
            n_g          = max(1, int(len(valid_tix) * PCT_GROUP))
            sv           = vol_col.sort_values()
            low_tix_all  = sv.index[:n_g].tolist()
            high_tix_all = sv.index[-n_g:].tolist()

            comp_list.append({
                'date'        : pred_date,
                'n_stocks'    : len(valid_tix),
                'eff_n'       : 1.0 / float((w**2).sum()) if (w**2).sum() > 0 else 0,
                'top10_share' : float(w.nlargest(10).sum()),
                'top1_weight' : float(w.max()),
                'top1_ticker' : w.idxmax(),
                'avg_vol'     : float((w * vol_col).sum()),
                'low_weight'  : float(w.reindex(low_tix_all).sum()),
                'high_weight' : float(w.reindex(high_tix_all).sum()),
                'turnover'    : turnover,
                'tc_cost'     : turnover * tc,
            })

            meta_list.append({'date': pred_date, 'Q': mu_meta, 'lam': lam})
            weights_history[pred_date] = w
            prev_w = w

        except Exception as e:
            err_list.append({'date': pred_date, 'error': str(e)})
            if len(err_list) <= 3:
                print(f'  [{name}] 에러 {pred_date.date()}: {e}')

    total_min = (time.time() - _t0) / 60
    ret_df  = pd.DataFrame(ret_list).set_index('date')  if ret_list  else pd.DataFrame()
    spy_df  = pd.DataFrame(spy_list).set_index('date')  if spy_list  else pd.DataFrame()
    comp_df = pd.DataFrame(comp_list).set_index('date') if comp_list else pd.DataFrame()
    meta_df = pd.DataFrame(meta_list).set_index('date') if meta_list else pd.DataFrame()
    wts_df  = pd.DataFrame(weights_history).T            if weights_history else pd.DataFrame()

    print(f'  → [{name}] 완료: {len(ret_df)}개월 성공 / {len(err_list)}개 에러 / {total_min:.1f}분 소요')

    return {
        'config'   : cfg,
        'ret'      : ret_df['ret']       if 'ret'       in ret_df.columns else pd.Series(dtype=float),
        'gross_ret': ret_df['gross_ret'] if 'gross_ret' in ret_df.columns else pd.Series(dtype=float),
        'spy_ret'  : spy_df['ret']       if 'ret'       in spy_df.columns else pd.Series(dtype=float),
        'weights'  : wts_df,
        'comp'     : comp_df,
        'meta'     : meta_df,
        'errors'   : err_list,
    }


print('walk_forward() 함수 정의 완료 (캐시 사용, Σ_BL 적용)')

walk_forward() 함수 정의 완료 (캐시 사용, Σ_BL 적용)


In [34]:
# ══════════════════════════════════════════════════════════════
# 전체 실험 실행 + 저장
# ══════════════════════════════════════════════════════════════

SKIP_IF_EXISTS = True   # True → 이미 저장된 실험은 재실행 않고 스킵

completed, skipped = [], []
_all_t0 = time.time()
run_list = [cfg for cfg in EXPERIMENTS
            if not (SKIP_IF_EXISTS and (RESULTS_DIR / f"{cfg['name']}.pkl").exists())
            and not (cfg.get('p_mode') == 'lstm_predicted' and not LSTM_AVAILABLE)
            and not (cfg.get('omega_mode') == 'rmse' and not LSTM_AVAILABLE)]

print(f'실행 예정: {len(run_list)}개 / 전체 {len(EXPERIMENTS)}개')
print(f'스킵 (기존 결과): {len(EXPERIMENTS) - len(run_list)}개\n')

for j, cfg in enumerate(EXPERIMENTS):
    name     = cfg['name']
    out_path = RESULTS_DIR / f'{name}.pkl'

    if cfg.get('p_mode') == 'lstm_predicted' and not LSTM_AVAILABLE:
        print(f'[SKIP] {name} — LSTM 예측 파일 없음')
        skipped.append(name)
        continue
    if cfg.get('omega_mode') == 'rmse' and not LSTM_AVAILABLE:
        print(f'[SKIP] {name} — LSTM 예측 파일 없음')
        skipped.append(name)
        continue
    if SKIP_IF_EXISTS and out_path.exists():
        print(f'[SKIP] {name} — 이미 저장됨')
        skipped.append(name)
        continue

    done_so_far = len(completed)
    remaining   = len(run_list) - done_so_far
    elapsed_all = time.time() - _all_t0
    avg_per_exp = elapsed_all / done_so_far if done_so_far > 0 else 0
    eta_all     = avg_per_exp * remaining

    print(f'\n[{done_so_far+1}/{len(run_list)}] {name}  |  '
          f'전체 경과 {elapsed_all/60:.1f}분  |  ETA {eta_all/60:.1f}분')

    result = walk_forward(cfg)

    with open(out_path, 'wb') as f:
        pickle.dump(result, f)

    completed.append(name)

total_elapsed = (time.time() - _all_t0) / 60
print(f'\n{"="*60}')
print(f'완료: {len(completed)}개 / 스킵: {len(skipped)}개 / 총 {total_elapsed:.1f}분')
print(f'저장 위치: {RESULTS_DIR}')

실행 예정: 0개 / 전체 175개
스킵 (기존 결과): 175개

[SKIP] baseline — 이미 저장됨
[SKIP] prior_eq — 이미 저장됨
[SKIP] p_rp — 이미 저장됨
[SKIP] p_eq — 이미 저장됨
[SKIP] p_vol_mcap — 이미 저장됨
[SKIP] capm_no_bl — 이미 저장됨
[SKIP] naive_lowvol — 이미 저장됨
[SKIP] mat_mcap_mcap_fix_he — 이미 저장됨
[SKIP] mat_mcap_eq_fix_he — 이미 저장됨
[SKIP] mat_mcap_rp_fix_he — 이미 저장됨
[SKIP] p_lstm_vol_mcap — 이미 저장됨
[SKIP] mat_eq_mcap_fix_he — 이미 저장됨
[SKIP] mat_eq_eq_fix_he — 이미 저장됨
[SKIP] mat_eq_rp_fix_he — 이미 저장됨
[SKIP] prior_eq_p_lstm_vol_mcap — 이미 저장됨
[SKIP] q_lambda — 이미 저장됨
[SKIP] mat_mcap_mcap_lam_he — 이미 저장됨
[SKIP] q_inv_lambda — 이미 저장됨
[SKIP] mat_mcap_mcap_inv_he — 이미 저장됨
[SKIP] q_raw_lam — 이미 저장됨
[SKIP] mat_eq_rp_lam_he — 이미 저장됨
[SKIP] mat_eq_mcap_inv_he — 이미 저장됨
[SKIP] mat_eq_mcap_lam_he — 이미 저장됨
[SKIP] q_ff3_paper — 이미 저장됨
[SKIP] q_ff3_paper_omega_paper — 이미 저장됨
[SKIP] omega_paper — 이미 저장됨
[SKIP] mat_mcap_mcap_fix_pap — 이미 저장됨
[SKIP] prior_eq_q_lambda — 이미 저장됨
[SKIP] prior_eq_q_raw_lam — 이미 저장됨
[SKIP] p_vol_mcap_q_lambda — 이미 저장됨
[SKIP] p_v

In [35]:
# ── 빠른 결과 확인 ────────────────────────────────────────────
rf_monthly  = rf_series.copy()
spy_monthly = spy_series.copy()

print('=' * 90)
print(f"{'실험명':<25} {'Sharpe':>7} {'Sortino':>8} {'CAGR':>7} {'변동성':>7} {'MDD':>8} {'Beta':>6} {'Alpha':>7}")
print('-' * 90)

for pkl_file in sorted(RESULTS_DIR.glob('*.pkl')):
    with open(pkl_file, 'rb') as f:
        res = pickle.load(f)
    ret = res['ret']
    if len(ret) == 0:
        continue
    m = compute_metrics(ret, rf_monthly, res['config']['name'], mkt_ret=spy_monthly)
    beta_str  = f"{m['beta']:>6.3f}"  if not (isinstance(m['beta'],  float) and np.isnan(m['beta']))  else '   N/A'
    alpha_str = f"{m['alpha']:>7.2%}" if not (isinstance(m['alpha'], float) and np.isnan(m['alpha'])) else '    N/A'
    print(f"{m['label']:<25} {m['sharpe']:>7.3f} {m['sortino']:>8.3f} "
          f"{m['cagr']:>7.2%} {m['vol']:>7.2%} {m['mdd']:>8.2%} "
          f"{beta_str} {alpha_str}")

print('=' * 90)

실험명                        Sharpe  Sortino    CAGR     변동성      MDD   Beta   Alpha
------------------------------------------------------------------------------------------
baseline                    1.106    1.726  13.37%  10.98%  -13.03% -0.136  13.95%
capm_no_bl                  0.899    1.268  14.78%  15.13%  -22.19% -0.149  15.52%
mat_eq_eq_fix_he            0.996    1.529  12.75%  11.58%  -14.01% -0.125  13.19%
mat_eq_eq_fix_pap           1.094    1.752  16.24%  13.59%  -14.13% -0.179  17.20%
mat_eq_eq_fix_rms           0.995    1.512  12.71%  11.36%  -14.04% -0.112  12.81%
mat_eq_eq_inv_he            0.977    1.333  12.71%  11.58%  -16.93% -0.128  13.01%
mat_eq_eq_inv_pap           1.066    1.605  15.88%  13.60%  -18.29% -0.178  16.83%
mat_eq_eq_inv_rms           1.000    1.358  12.97%  11.58%  -16.95% -0.122  13.19%
mat_eq_eq_lam_he            0.934    1.656  12.34%  11.71%  -13.26% -0.121  12.55%
mat_eq_eq_lam_pap           1.098    1.844  16.24%  13.53%  -13.58% -0.176  17.

# ── H. 서브기간 Sharpe 매트릭스 + 히트맵 ─────────────────────
## 전체 실험을 3개 서브기간(2010-14 / 2015-19 / 2020-24)에서 Sharpe 비교

In [36]:
# ── H. 서브기간 Sharpe 매트릭스 + 히트맵 ──────────────────────
import matplotlib.pyplot as plt
import platform

# 한글 폰트
if platform.system() == 'Darwin':
    plt.rcParams['font.family']     = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['AppleGothic', 'Arial Unicode MS', 'DejaVu Sans']
elif platform.system() == 'Windows':
    plt.rcParams['font.family']     = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Malgun Gothic', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

OUT_DIR = BASE_DIR / 'outputs'
OUT_DIR.mkdir(exist_ok=True)

# ── 모든 결과 로드 ────────────────────────────────────────────
loaded = {}
for pkl_file in sorted(RESULTS_DIR.glob('*.pkl')):
    with open(pkl_file, 'rb') as f:
        res = pickle.load(f)
    if len(res.get('ret', pd.Series())) > 0:
        loaded[pkl_file.stem] = res

rf      = rf_series.copy()
spy_ret = spy_series.copy()

# 전체기간 Sharpe (히트맵 정렬용)
def calc(name):
    r    = loaded[name]['ret'].dropna()
    rf_a = rf.reindex(r.index).fillna(0)
    exc  = r - rf_a
    vol  = r.std() * np.sqrt(12)
    return {'sharpe': (exc.mean() * 12) / vol if vol > 0 else np.nan}

# ── H1. 서브기간 정의 및 성과 계산 ────────────────────────────
PERIODS = {
    '2010-2014': ('2010-01-01', '2014-12-31'),
    '2015-2019': ('2015-01-01', '2019-12-31'),
    '2020-2024': ('2020-01-01', '2024-12-31'),
}

def calc_sub(name, start, end):
    r    = loaded[name]['ret'].dropna()
    r    = r[(r.index >= start) & (r.index <= end)]
    if len(r) < 6:
        return None
    rf_a = rf.reindex(r.index).fillna(0)
    exc  = r - rf_a
    ann  = exc.mean() * 12
    vol  = r.std() * np.sqrt(12)
    sh   = ann / vol if vol > 0 else np.nan
    cum  = (1 + r).cumprod()
    mdd  = ((cum / cum.cummax()) - 1).min()
    dsd  = r[r < 0].std() * np.sqrt(12)
    sor  = ann / dsd if dsd > 0 else np.nan
    cagr = r.mean() * 12
    return dict(sharpe=sh, cagr=cagr, vol=vol, mdd=mdd, sortino=sor)

def calc_spy_sub(start, end):
    r    = spy_ret[(spy_ret.index >= start) & (spy_ret.index <= end)]
    rf_a = rf.reindex(r.index).fillna(0)
    exc  = r - rf_a
    ann  = exc.mean() * 12
    vol  = r.std() * np.sqrt(12)
    sh   = ann / vol if vol > 0 else np.nan
    return dict(sharpe=sh, cagr=r.mean()*12, vol=vol)

all_exps   = sorted(loaded.keys())
sharpe_mat = {}
for period, (s, e) in PERIODS.items():
    sharpe_mat[period] = {n: (calc_sub(n, s, e)['sharpe'] if calc_sub(n, s, e) else np.nan) for n in all_exps}

df_sub = pd.DataFrame(sharpe_mat).T   # periods × experiments

print('=== 서브기간별 Sharpe 매트릭스 ===')
print(df_sub.round(3).to_string())
print()
print('=== SPY 서브기간 Sharpe ===')
for period, (s, e) in PERIODS.items():
    ms_sub = calc_spy_sub(s, e)
    print(f"  {period}: {ms_sub['sharpe']:.3f}")

# ── H2. Sharpe 히트맵 (전체 실험 × 서브기간) ───────────────────
fig, ax = plt.subplots(figsize=(20, 5))

full_sharpe = {n: calc(n)['sharpe'] for n in all_exps}
exp_order   = sorted(all_exps, key=lambda n: -full_sharpe[n] if not np.isnan(full_sharpe[n]) else 999)
heat_data   = df_sub[exp_order]

spy_shs = {p: calc_spy_sub(s, e)['sharpe'] for p, (s, e) in PERIODS.items()}

im = ax.imshow(heat_data.values, aspect='auto', cmap='RdYlGn',
               vmin=0.0, vmax=2.5, interpolation='nearest')

ax.set_xticks(range(len(exp_order)))
ax.set_xticklabels(exp_order, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(PERIODS)))
ax.set_yticklabels(list(PERIODS.keys()), fontsize=10)

# 셀 값 표시
for i, period in enumerate(PERIODS.keys()):
    for j, exp in enumerate(exp_order):
        v = heat_data.loc[period, exp]
        txt_color = 'white' if v < 0.6 or v > 2.0 else 'black'
        if not np.isnan(v):
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    fontsize=7, color=txt_color, fontweight='bold')

# SPY 기준선 표시
spy_text = '  '.join([f"{p}: SPY={v:.2f}" for p, v in spy_shs.items()])
ax.set_xlabel(f'실험명 (전체기간 Sharpe 내림차순)   |   {spy_text}', fontsize=9)
ax.set_title('H2. 서브기간별 Sharpe 히트맵  (녹색=높음, 적색=낮음)', fontweight='bold', fontsize=12)
plt.colorbar(im, ax=ax, label='Sharpe', shrink=0.8)
plt.tight_layout()
plt.savefig(OUT_DIR / 'H_subperiod_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✓ 저장: {OUT_DIR / "H_subperiod_heatmap.png"}')

=== 서브기간별 Sharpe 매트릭스 ===
           baseline  capm_no_bl  mat_eq_eq_fix_he  mat_eq_eq_fix_pap  mat_eq_eq_fix_rms  mat_eq_eq_inv_he  mat_eq_eq_inv_pap  mat_eq_eq_inv_rms  mat_eq_eq_lam_he  mat_eq_eq_lam_pap  mat_eq_eq_lam_rms  mat_eq_eq_raw_he  mat_eq_eq_raw_pap  mat_eq_eq_raw_rms  mat_eq_eq_vsp_he  mat_eq_eq_vsp_pap  mat_eq_eq_vsp_rms  mat_eq_mcap_fix_he  mat_eq_mcap_fix_pap  mat_eq_mcap_fix_rms  mat_eq_mcap_inv_he  mat_eq_mcap_inv_pap  mat_eq_mcap_inv_rms  mat_eq_mcap_lam_he  mat_eq_mcap_lam_pap  mat_eq_mcap_lam_rms  mat_eq_mcap_raw_he  mat_eq_mcap_raw_pap  mat_eq_mcap_raw_rms  mat_eq_mcap_vsp_he  mat_eq_mcap_vsp_pap  mat_eq_mcap_vsp_rms  mat_eq_rp_fix_he  mat_eq_rp_fix_pap  mat_eq_rp_fix_rms  mat_eq_rp_inv_he  mat_eq_rp_inv_pap  mat_eq_rp_inv_rms  mat_eq_rp_lam_he  mat_eq_rp_lam_pap  mat_eq_rp_lam_rms  mat_eq_rp_raw_he  mat_eq_rp_raw_pap  mat_eq_rp_raw_rms  mat_eq_rp_vsp_he  mat_eq_rp_vsp_pap  mat_eq_rp_vsp_rms  mat_mcap_eq_fix_he  mat_mcap_eq_fix_pap  mat_mcap_eq_fix_rms  mat_mcap_

# ── I. 레짐 기반 서브기간 성과 (Sortino > MDD > Sharpe + prior별 분리) ─

## 5개 레짐 (데이터 기반: SPY 누적/12M/drawdown 검증)
- **R1 회복** 2010-01~2014-12 (60M, Post-GFC)
- **R2 확장** 2015-01~2018-12 (48M, 저변동 + 18Q4)
- **R3 COVID** 2019-01~2020-12 (24M, Pre-COVID + 크래시)
- **R4 베어** 2021-01~2022-12 (24M, 22 베어)
- **R5 AI 랠리** 2023-01~2024-12 (23M, Mag 7 강세)

## 평가 우선순위 = Sortino > MDD > Sharpe
변동성 다루는 프로젝트라 하방위험 우선. Sortino는 상승 변동성을 페널티하지 않으므로 저위험 anomaly에 더 정확.

In [37]:
# ── I. 레짐 기반 서브기간 성과 (Sortino + 3 prior 히트맵) ─────
REGIMES = [
    ('R1 회복',     '2010-01-01', '2014-12-31', '#2ecc71'),
    ('R2 확장',     '2015-01-01', '2018-12-31', '#3498db'),
    ('R3 COVID',    '2019-01-01', '2020-12-31', '#e74c3c'),
    ('R4 베어',     '2021-01-01', '2022-12-31', '#f39c12'),
    ('R5 AI 랠리',  '2023-01-01', '2024-12-31', '#9b59b6'),
]
REGIME_LABELS = [r[0] for r in REGIMES]

def metrics_full(r):
    r = r.dropna()
    if len(r) < 6: return {'sharpe': np.nan, 'sortino': np.nan, 'mdd': np.nan}
    mean_ann = r.mean() * 12
    vol_ann  = r.std() * np.sqrt(12)
    downside = r[r < 0].std() * np.sqrt(12)
    cum = (1 + r).cumprod()
    return {
        'sharpe' : mean_ann/vol_ann if vol_ann > 0 else np.nan,
        'sortino': mean_ann/downside if downside > 0 else np.nan,
        'mdd'    : float((cum/cum.cummax() - 1).min()),
    }

# ── I1. 모든 실험 × 레짐 × Sortino/MDD/Sharpe ──────────────────
I_records = []
for name, res in loaded.items():
    cfg = res.get('config', {})
    r = res['ret']
    rec = {'name': name, 'prior': cfg.get('prior', 'capm_mcap')}
    for label, s, e, _ in REGIMES:
        m = metrics_full(r.loc[s:e])
        for k in ['sortino', 'mdd', 'sharpe']:
            rec[f'{label}_{k}'] = m[k]
    m_full = metrics_full(r)
    for k in ['sortino', 'mdd', 'sharpe']:
        rec[f'full_{k}'] = m_full[k]
    I_records.append(rec)
df_I = pd.DataFrame(I_records)

# SPY 레짐별 baseline
spy_metrics = {}
for label, s, e, _ in REGIMES:
    spy_metrics[label] = metrics_full(spy_ret.loc[s:e])
spy_metrics['full'] = metrics_full(spy_ret)

print('=== SPY 레짐별 Sortino / MDD / Sharpe ===')
for label in REGIME_LABELS + ['full']:
    m = spy_metrics[label]
    print(f"  {label:12s}  Sortino={m['sortino']:.3f}  MDD={m['mdd']:+.2%}  Sharpe={m['sharpe']:.3f}")

# ── I2. prior별 히트맵 3장 (mcap / eq / rp) ───────────────────
def heatmap_for_prior(df_I, prior_val, title_suffix, save_name):
    sub = df_I[df_I['prior'] == prior_val].copy()
    if len(sub) == 0:
        print(f'  prior={prior_val} 데이터 없음, 스킵')
        return
    sub = sub.sort_values('full_sortino', ascending=False)
    n = len(sub)
    cols = [f'{l}_sortino' for l in REGIME_LABELS] + ['full_sortino']
    mat = sub[cols].values
    fig, ax = plt.subplots(figsize=(11, max(5, n*0.32)))
    im = ax.imshow(mat, cmap='RdYlGn', vmin=0, vmax=3.0, aspect='auto')
    ax.set_yticks(range(n)); ax.set_yticklabels(sub['name'], fontsize=7)
    ax.set_xticks(range(6)); ax.set_xticklabels(REGIME_LABELS + ['full'], rotation=20, ha='right')
    for i in range(n):
        for j in range(6):
            v = mat[i, j]
            if pd.notna(v):
                ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                        fontsize=7, color='black' if 0.5 < v < 2.5 else 'white')
    spy_line = ' | '.join(f"{l}: SPY={spy_metrics[l]['sortino']:.2f}" for l in REGIME_LABELS)
    ax.set_xlabel(f'Sortino — {spy_line}', fontsize=8)
    ax.set_title(f'I2. {title_suffix} | Sortino 히트맵 (full Sortino 정렬)', fontweight='bold')
    plt.colorbar(im, label='Sortino'); plt.tight_layout()
    plt.savefig(OUT_DIR / save_name, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'  ✓ {save_name}: {n} 실험')

for prior_val, suffix, fname in [
    ('capm_mcap', 'prior=mcap (시총가중)',     'I_heatmap_prior_mcap.png'),
    ('capm_eq',   'prior=eq (1/N 균등)',        'I_heatmap_prior_eq.png'),
    ('capm_rp',   'prior=rp (1/σ Risk Parity)', 'I_heatmap_prior_rp.png'),
]:
    heatmap_for_prior(df_I, prior_val, suffix, fname)

# ── I3. SPY 4+/5 레짐 Sortino 초과 winners ─────────────────────
def pass_sortino_count(row):
    return sum(1 for l in REGIME_LABELS if row[f'{l}_sortino'] > spy_metrics[l]['sortino'])
df_I['pass_sortino'] = df_I.apply(pass_sortino_count, axis=1)
winners = df_I[df_I['pass_sortino'] >= 4].sort_values(
    by=['full_sortino', 'full_mdd', 'full_sharpe'], ascending=[False, False, False])

print(f'\n=== Tier 4 (실용): Sortino 4+/5 레짐 SPY 초과 ({len(winners)}개) ===')
if len(winners) > 0:
    cols_show = ['name', 'prior', 'pass_sortino', 'full_sortino', 'full_mdd', 'full_sharpe']
    print(winners[cols_show].head(20).round(3).to_string(index=False))

winners.to_csv(OUT_DIR / 'I_winners_sortino_4of5.csv', index=False)
df_I.to_csv(OUT_DIR / 'I_all_regime_metrics.csv', index=False)
print(f'\n✓ 저장: I_heatmap_prior_*.png, I_winners_sortino_4of5.csv, I_all_regime_metrics.csv')


=== SPY 레짐별 Sortino / MDD / Sharpe ===
  R1 회복         Sortino=1.875  MDD=-16.22%  Sharpe=1.169
  R2 확장         Sortino=0.877  MDD=-13.53%  Sharpe=0.655
  R3 COVID      Sortino=1.627  MDD=-19.45%  Sharpe=1.209
  R4 베어         Sortino=0.413  MDD=-23.93%  Sharpe=0.229
  R5 AI 랠리      Sortino=5.115  MDD=-8.33%  Sharpe=1.883
  full          Sortino=1.028  MDD=-50.78%  Sharpe=0.765
  ✓ I_heatmap_prior_mcap.png: 67 실험
  ✓ I_heatmap_prior_eq.png: 61 실험
  ✓ I_heatmap_prior_rp.png: 47 실험

=== Tier 4 (실용): Sortino 4+/5 레짐 SPY 초과 (79개) ===
                                         name     prior  pass_sortino  full_sortino  full_mdd  full_sharpe
                                    q_raw_lam capm_mcap             4         2.169    -0.127        1.185
                          mat_eq_mcap_raw_rms   capm_eq             4         2.111    -0.130        1.162
                        mat_mcap_mcap_raw_rms capm_mcap             4         2.111    -0.125        1.175
                                     